# Netflix Content Analytics — Exploratory Data Analysis

This notebook explores the Netflix titles catalog: what kind of content Netflix has added over time, where it comes from, how it's rated, and how long it runs.

**Dataset**: Kaggle ["Netflix Movies and TV Shows"](https://www.kaggle.com/datasets/shivamb/netflix-shows) (`netflix_titles.csv`). If the CSV isn't present in `data/`, a synthetic sample dataset is generated automatically so this notebook always runs end-to-end — see `src/generate_sample_data.py`.

**Sections**
1. Load & clean the data
2. Missing values
3. Content type mix (Movies vs. TV Shows)
4. Growth over time
5. Countries & genres
6. Ratings & duration
7. Directors
8. Key takeaways

In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src import analysis
from src.data_loader import load_clean, load_raw

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

## 1. Load & clean the data

In [ ]:
raw = load_raw()
print(raw.shape)
raw.head()

In [ ]:
df = load_clean()
df.info()

## 2. Missing values

In [ ]:
missing = raw.isna().mean().sort_values(ascending=False) * 100
missing[missing > 0].round(1).to_frame("pct_missing")

`director`, `cast`, and `country` are the columns most likely to have gaps. These are filled with `"Unknown"` during cleaning (see `src/data_loader.clean`) rather than dropped, so we don't lose rows for downstream counts that don't depend on the missing field.

## 3. Content type mix

In [ ]:
type_counts = analysis.content_type_counts(df)
display(type_counts)

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(type_counts["count"], labels=type_counts["type"], autopct="%1.0f%%",
       colors=["#2a78d6", "#eb6834"])
ax.set_title("Movies vs. TV Shows")
plt.show()

## 4. Growth over time

In [ ]:
added = analysis.titles_added_per_year(df)
pivot = added.pivot(index="year_added", columns="type", values="count").fillna(0)

ax = pivot.plot(kind="bar", stacked=True, figsize=(10, 5),
                 color=["#2a78d6", "#eb6834"])
ax.set_title("Titles Added to Netflix per Year")
ax.set_xlabel("Year added")
ax.set_ylabel("Titles")
plt.tight_layout()
plt.show()

In [ ]:
trend = analysis.release_year_trend(df)
pivot2 = trend.pivot(index="release_year", columns="type", values="count").fillna(0)

ax = pivot2.plot(figsize=(10, 5), marker="o", color=["#2a78d6", "#eb6834"])
ax.set_title("Titles by Release Year")
ax.set_xlabel("Release year")
ax.set_ylabel("Titles")
plt.tight_layout()
plt.show()

## 5. Countries & genres

In [ ]:
countries = analysis.top_countries(df, n=10).sort_values("count")
ax = countries.plot.barh(x="country", y="count", figsize=(8, 5), color="#1baf7a", legend=False)
ax.set_title("Top 10 Countries by Title Count")
ax.set_xlabel("Titles")
plt.tight_layout()
plt.show()

In [ ]:
genres = analysis.top_genres(df, n=10).sort_values("count")
ax = genres.plot.barh(x="genre", y="count", figsize=(8, 5), color="#4a3aa7", legend=False)
ax.set_title("Top 10 Genres")
ax.set_xlabel("Titles")
plt.tight_layout()
plt.show()

## 6. Ratings & duration

In [ ]:
ratings = analysis.rating_distribution(df).sort_values("count")
ax = ratings.plot.barh(x="rating", y="count", figsize=(8, 6), color="#2a78d6", legend=False)
ax.set_title("Content Ratings")
ax.set_xlabel("Titles")
plt.tight_layout()
plt.show()

In [ ]:
analysis.duration_summary(df).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df.loc[df["type"] == "Movie", "duration_minutes"].dropna(), bins=30,
             color="#2a78d6", ax=ax)
ax.set_title("Movie Duration Distribution (minutes)")
plt.tight_layout()
plt.show()

## 7. Directors

In [ ]:
directors = analysis.top_directors(df, n=10).sort_values("count")
ax = directors.plot.barh(x="director", y="count", figsize=(8, 5), color="#e87ba4", legend=False)
ax.set_title("Top 10 Directors by Title Count")
ax.set_xlabel("Titles")
plt.tight_layout()
plt.show()

## 8. Key takeaways

- Netflix's catalog skews toward one content type over the other (see the pie chart in §3) — quantify the split for your specific data pull.
- Additions to the platform (`date_added`) accelerated sharply in the mid-to-late 2010s, well after most titles' original `release_year` — most of the catalog is licensed back-catalog, not day-one originals.
- Content is concentrated in a handful of countries and genres (§5) — a long tail of countries/genres each contribute only one or two titles.
- Movie runtimes cluster tightly around a typical feature length; TV show season counts are heavily right-skewed (most shows run 1–2 seasons, a few run much longer).

> Replace this section with the actual numbers once you run this notebook against the real Kaggle CSV — the synthetic fallback dataset is for structural testing only and does not reflect real-world patterns.